1. Run the Apriori Notebook Shared by me on basket dataset using different Support and confidence values.


In [12]:
from collections import Counter,OrderedDict

In [27]:
# 3rd approach 
from itertools import combinations

def genSubsets(l):
    result = []
    for i in range(len(l) + 1):
        for comb in combinations(l, i):
            result.append(list(comb))
    return result

In [28]:

    
def initPass(txList): # list of transactions, most possibly a dict
    allTx = [item for tx in txList for item in tx]
    allTx.sort()
    cntr =  OrderedDict()
    for tx in allTx:
        cntr[tx] = cntr.get(tx,0) + 1

    return cntr



In [29]:
# Apriori Algorithm Implementation
def genCandidate(Fk1): #Fk1 indicates F(k-1), it is a list of lists
    Ck = []
    k1  = len(Fk1[0])

    # COMBINE STEP
    for i in range(len(Fk1)-1):
        for j in range(i+1,len(Fk1)):
            f1,f2 = Fk1[i],Fk1[j]

            if f1[:len(f1)-1] == f2[:len(f2)-1] and f1[-1] < f2[-1]:
                tempC = f1 + [f2[-1]]

                # PRUNING STEP
                subset = genSubsets(tempC)
                appendSts = True
                for item in subset:
                    if len(item) == k1 and item not in Fk1:
                        appendSts = False
                if appendSts:
                    Ck.append(tempC)              
    return Ck

In [30]:
def searchInT(t,candid):
    found = True
    for eachCandid in candid:
        if eachCandid not in t:
            found = False

    return found


In [31]:
def apriori(T,minSup):
    finalSet = []
    c1 = initPass(T)
    f = [[item] for item in c1.keys() if c1[item]/len(T) >= minSup] # f1
    #print(f)
    for item in f:
        finalSet.append(item)

    while len(f) != 0:
        Ck = genCandidate(f)
        # print("Ck")
        # print(Ck)
        freqDict = {}
        for t in T:
            for candidate in Ck:
                if searchInT(t,candidate):
                    freqDict[tuple(candidate)] = freqDict.get(tuple(candidate),0) + 1
        # print("freqDict")
        # print(freqDict)
        f = []
        for c in freqDict.keys():
            if freqDict[c]/len(T)>= minSup:
                f.append(list(c))

        # print("f")
        # print(f)
        if len(f) != 0:
            f = sorted(f,key=lambda x : (len(x),*x))
            for item in f:
                finalSet.append(item)
    # print(finalSet)
    return finalSet

In [43]:
import pyECLAT
import pandas as pd

data = pd.read_csv(r"D:\College_work\ML\LAB_6\basket.csv")
print(data.head())

     1         2                 3                4      5   6
0  LBE  Brooklyn             11204              NaN    NaN NaN
1  MBE       WBE             BLACK  Cambria Heights  11411 NaN
2  MBE     BLACK  Yorktown Heights            10598    NaN NaN
3  MBE     BLACK        Long Beach            11561    NaN NaN
4  MBE     ASIAN          Brooklyn            11235    NaN NaN


In [44]:
data

,1,2,3,4,5,6
0,LBE,Brooklyn,11204,NaN,NaN,NaN
1,MBE,WBE,BLACK,Cambria Heights,11411,NaN
2,MBE,BLACK,Yorktown Heights,10598,NaN,NaN
3,MBE,BLACK,Long Beach,11561,NaN,NaN
4,MBE,ASIAN,Brooklyn,11235,NaN,NaN
...,...,...,...,...,...,...
1415,WBE,NON-MINORITY,New York,10023,NaN,NaN
1416,MBE,ASIAN,Valley Stream,11580,NaN,NaN
1417,MBE,BLACK,Brooklyn,11214,NaN,NaN
1418,LBE,New York,10016,NaN,NaN,NaN


In [45]:

T3 = data.values.tolist()
print(T3)

[['LBE', 'Brooklyn', '11204', nan, nan, nan], ['MBE', 'WBE', 'BLACK', 'Cambria Heights', '11411', nan], ['MBE', 'BLACK', 'Yorktown Heights', '10598', nan, nan], ['MBE', 'BLACK', 'Long Beach', '11561', nan, nan], ['MBE', 'ASIAN', 'Brooklyn', '11235', nan, nan], ['MBE', 'WBE', 'ASIAN', 'New York', '10010', nan], ['MBE', 'ASIAN', 'New York', '10026', nan, nan], ['MBE', 'BLACK', 'New York', '10026', nan, nan], ['MBE', 'HISPANIC', 'New York', '10034', nan, nan], ['MBE', 'WBE', 'BLACK', 'Staten Island', '10303', nan], ['MBE', 'ASIAN', 'New York', '10018', nan, nan], ['MBE', 'WBE', 'HISPANIC', 'New York', '10034', nan], ['MBE', 'WBE', 'ASIAN', 'New York', '10013', nan], ['MBE', 'BLACK', 'Jamaica', '11434', nan, nan], ['WBE', 'NON-MINORITY', 'New York', '10022', nan, nan], ['MBE', 'BLACK', 'Staten Island', '10304', nan, nan], ['MBE', 'BLACK', 'Bronx', '10454', nan, nan], ['WBE', 'NON-MINORITY', 'New Rochelle', '10801', nan, nan], ['WBE', 'NON-MINORITY', 'Staten Island', '10301', nan, nan], ['W

In [46]:
data = data.fillna('') 

In [47]:
# print(initPass())

# we are looking for itemSETS
# we do not want to have any individual products returned
min_n_products = 2

# we want to set min support to 7
# but we have to express it as a percentage
min_support = 7/len(T3)

# we have no limit on the size of association rules
# so we set it to the longest transaction
max_length = max([len(x) for x in T3])

In [49]:
!pip install pyECLAT

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
from pyECLAT import ECLAT

# create an instance of eclat
my_eclat = ECLAT(data=data, verbose=True)

# fit the algorithm
rule_indices, rule_supports = my_eclat.fit(min_support=min_support,
                                        min_combination=min_n_products,
                                        max_combination=max_length)

  0%|          | 0/728 [00:00<?, ?it/s]


KeyError: 0

In [42]:
import pandas as pd
from pyECLAT import ECLAT

# Step 1: Load dataset
data = pd.read_csv(r"D:\College_work\ML\LAB_6\basket.csv")

# Step 2: Clean data
data = data.fillna('')

# Step 3: Different support values
support_values = [0.1, 0.2, 0.3, 0.4, 0.5]

for sup in support_values:
    print(f"\n===== Support: {sup} =====")
    
    # Initialize ECLAT
    eclat = ECLAT(data=data, verbose=False)
    
    # Fit model
    rule_indices, rule_supports = eclat.fit(min_support=sup)
    
    # Output
    print("Number of Frequent Itemsets:", len(rule_supports))
    print(rule_supports)


===== Support: 0.1 =====


KeyError: 0

2. What is maximum size of rule that can be created?


3. At what Confidence value, Minimum number of rules are generated.